In [ ]:
# ============================================================
# CELL 1 — Install Dependencies
# ============================================================
!pip install -q anthropic pyspark fpdf2 matplotlib seaborn scikit-learn

In [ ]:
# ============================================================
# CELL 2 — Set Your Anthropic API Key
# ============================================================
import os
from google.colab import userdata

try:
    os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
    print(" API key loaded from Colab Secrets")
except:
    # Option B: Paste key directly
    os.environ["ANTHROPIC_API_KEY"] = "sk-ant-PASTE_YOUR_KEY_HERE"
    print(" Using hardcoded API key — use Secrets for production")

In [ ]:
# ============================================================
# CELL 3 — Full Pipeline Code
# ============================================================

import os, sys, json, time, warnings
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler, LabelEncoder
from anthropic import Anthropic
from fpdf import FPDF
warnings.filterwarnings('ignore')

os.makedirs('outputs/charts', exist_ok=True)
client = Anthropic()
print(" All imports successful")


# ══════════════════════════════════════════════════════════════
# STAGE 1 — LLM Dataset Generation
# ══════════════════════════════════════════════════════════════

DATASET_TOOL = {
    "name": "generate_dataset",
    "description": "Generate a synthetic retail sales dataset as a list of records.",
    "input_schema": {
        "type": "object",
        "properties": {
            "records": {
                "type": "array",
                "items": {
                    "type": "object",
                    "properties": {
                        "order_id":       {"type": "string"},
                        "customer_id":    {"type": "string"},
                        "customer_name":  {"type": "string"},
                        "age":            {"type": "integer"},
                        "gender":         {"type": "string", "enum": ["Male","Female","Other"]},
                        "city":           {"type": "string"},
                        "category":       {"type": "string", "enum": ["Electronics","Clothing","Home & Garden","Sports","Books"]},
                        "product_name":   {"type": "string"},
                        "quantity":       {"type": "integer"},
                        "unit_price":     {"type": "number"},
                        "discount_pct":   {"type": "number"},
                        "order_date":     {"type": "string"},
                        "payment_method":{"type": "string", "enum": ["Credit Card","Debit Card","UPI","Net Banking","Cash"]},
                        "status":         {"type": "string", "enum": ["Delivered","Pending","Cancelled","Returned"]},
                        "rating":         {"type": "number"}
                    },
                    "required": ["order_id","customer_id","customer_name","age","gender","city",
                                 "category","product_name","quantity","unit_price","discount_pct",
                                 "order_date","payment_method","status","rating"]
                }
            }
        },
        "required": ["records"]
    }
}

def generate_dataset(prompt, num_records=200):
    print(f"\n{'='*60}")
    print(f"  STAGE 1 — Generating {num_records} records via LLM...")
    print(f"{'='*60}")
    all_records = []
    batch_size = 50
    batches = num_records // batch_size
    for i in range(batches):
        print(f"  Batch {i+1}/{batches}...")
        response = client.messages.create(
            model="claude-sonnet-4-20250514",
            max_tokens=8000,
            tools=[DATASET_TOOL],
            tool_choice={"type": "tool", "name": "generate_dataset"},
            messages=[{"role": "user", "content":
                f"""{prompt}\nGenerate {batch_size} realistic Indian retail sales records.
Use Indian cities (Mumbai, Delhi, Bangalore, Chennai, Hyderabad, Pune, Kolkata).
Orders across Jan-Dec 2024. Include ~5% missing values for realism.
Batch {i+1} — use order IDs starting ORD{(i*batch_size)+1:04d}."""}]
        )
        for block in response.content:
            if block.type == "tool_use":
                all_records.extend(block.input["records"])
    df = pd.DataFrame(all_records)
    print(f" Generated {len(df)} rows × {len(df.columns)} cols")
    return df


# ══════════════════════════════════════════════════════════════
# STAGE 2 — Agentic Data Cleaning
# ══════════════════════════════════════════════════════════════

CLEANING_TOOLS = [
    {"name": "drop_duplicates",
     "description": "Remove duplicate rows.",
     "input_schema": {"type": "object", "properties": {}, "required": []}},
    {"name": "fill_missing_numeric",
     "description": "Fill missing numeric values with median.",
     "input_schema": {"type": "object",
                      "properties": {"columns": {"type": "array", "items": {"type": "string"}}},
                      "required": ["columns"]}},
    {"name": "fill_missing_categorical",
     "description": "Fill missing categorical values with mode or constant.",
     "input_schema": {"type": "object",
                      "properties": {
                          "columns": {"type": "array", "items": {"type": "string"}},
                          "strategy": {"type": "string", "enum": ["mode", "constant"]},
                          "fill_value": {"type": "string"}
                      }, "required": ["columns", "strategy"]}},
    {"name": "fix_data_types",
     "description": "Convert columns to correct data types.",
     "input_schema": {"type": "object",
                      "properties": {"conversions": {"type": "array",
                          "items": {"type": "object",
                                    "properties": {"column": {"type": "string"},
                                                   "dtype": {"type": "string"}}}}},
                      "required": ["conversions"]}},
    {"name": "cap_outliers",
     "description": "Cap outliers using IQR method.",
     "input_schema": {"type": "object",
                      "properties": {
                          "columns": {"type": "array", "items": {"type": "string"}},
                          "multiplier": {"type": "number"}
                      }, "required": ["columns"]}},
    {"name": "cleaning_done",
     "description": "Signal cleaning is complete.",
     "input_schema": {"type": "object",
                      "properties": {"summary": {"type": "string"}},
                      "required": ["summary"]}}
]

def _df_info(df):
    return json.dumps({
        "shape": df.shape,
        "dtypes": df.dtypes.astype(str).to_dict(),
        "missing_counts": df.isnull().sum().to_dict(),
        "duplicates": int(df.duplicated().sum()),
        "sample": df.head(3).to_dict(orient="records")
    }, default=str)

def _apply_cleaning_tool(df, name, inp):
    if name == "drop_duplicates":
        b = len(df); df = df.drop_duplicates()
        return df, f"Dropped {b-len(df)} duplicates"
    elif name == "fill_missing_numeric":
        for c in inp["columns"]:
            if c in df.columns: df[c] = df[c].fillna(df[c].median())
        return df, f"Filled numeric nulls: {inp['columns']}"
    elif name == "fill_missing_categorical":
        for c in inp["columns"]:
            if c in df.columns:
                v = df[c].mode()[0] if inp["strategy"]=="mode" and not df[c].mode().empty else inp.get("fill_value","Unknown")
                df[c] = df[c].fillna(v)
        return df, f"Filled categorical nulls: {inp['columns']}"
    elif name == "fix_data_types":
        for conv in inp["conversions"]:
            c, dt = conv["column"], conv["dtype"]
            if c in df.columns:
                try:
                    if dt in ("numeric","float"): df[c] = pd.to_numeric(df[c], errors="coerce")
                    elif dt == "integer": df[c] = pd.to_numeric(df[c], errors="coerce").astype("Int64")
                    elif dt == "datetime": df[c] = pd.to_datetime(df[c], errors="coerce")
                except: pass
        return df, f"Fixed dtypes for {[c['column'] for c in inp['conversions']]}"
    elif name == "cap_outliers":
        m = inp.get("multiplier", 1.5)
        for c in inp["columns"]:
            if c in df.columns and pd.api.types.is_numeric_dtype(df[c]):
                Q1,Q3 = df[c].quantile(0.25), df[c].quantile(0.75)
                df[c] = df[c].clip(Q1-m*(Q3-Q1), Q3+m*(Q3-Q1))
        return df, f"Capped outliers: {inp['columns']}"
    return df, f"Unknown: {name}"

def clean_dataset(df):
    print(f"\n{'='*60}")
    print("  STAGE 2 — Agentic Data Cleaning...")
    print(f"{'='*60}")
    messages = [{"role": "user", "content":
        f"""You are a data cleaning expert. Clean this dataset step by step using tools.

Dataset info:
{_df_info(df)}

Steps to perform in order:
1. fix_data_types: order_date→datetime, quantity/unit_price/discount_pct/rating→numeric
2. drop_duplicates
3. fill_missing_numeric for: quantity, unit_price, discount_pct, rating, age
4. fill_missing_categorical for: gender, city, category, payment_method, status
5. cap_outliers for: quantity, unit_price, discount_pct
6. cleaning_done with summary

Use one tool per response."""}]
    for i in range(15):
        resp = client.messages.create(
            model="claude-sonnet-4-20250514", max_tokens=1500,
            tools=CLEANING_TOOLS, messages=messages
        )
        messages.append({"role": "assistant", "content": resp.content})
        tool_results = []
        done = False
        for block in resp.content:
            if block.type == "tool_use":
                if block.name == "cleaning_done":
                    print(f"  Cleaning done: {block.input['summary'][:100]}")
                    done = True
                    tool_results.append({"type":"tool_result","tool_use_id":block.id,"content":"Done."})
                else:
                    df, log = _apply_cleaning_tool(df, block.name, block.input)
                    print(f"  [tool] {block.name}: {log}")
                    tool_results.append({"type":"tool_result","tool_use_id":block.id,
                                         "content":f"Done. {log}\nUpdated info:\n{_df_info(df)}"})
        if tool_results:
            messages.append({"role":"user","content":tool_results})
        if done or resp.stop_reason == "end_turn": break
    df.columns = [c.lower().replace(" ","_").replace("-","_") for c in df.columns]
    print(f"  Cleaned shape: {df.shape}")
    return df


# ══════════════════════════════════════════════════════════════
# STAGE 3 — Data Transformation
# ══════════════════════════════════════════════════════════════

def transform_dataset(df):
    print(f"\n{'='*60}")
    print("  STAGE 3 — Feature Engineering, Encoding, Scaling...")
    print(f"{'='*60}")
    df = df.copy()
    df["order_date"] = pd.to_datetime(df["order_date"], errors="coerce")
    df["quantity"]    = pd.to_numeric(df["quantity"],    errors="coerce").fillna(1)
    df["unit_price"]  = pd.to_numeric(df["unit_price"],  errors="coerce").fillna(0)
    df["discount_pct"]= pd.to_numeric(df["discount_pct"],errors="coerce").fillna(0)
    df["rating"]      = pd.to_numeric(df["rating"],      errors="coerce").fillna(3)
    df["age"]         = pd.to_numeric(df["age"],         errors="coerce").fillna(30)

    # --- Feature Engineering ---
    df["revenue"]       = df["quantity"] * df["unit_price"] * (1 - df["discount_pct"]/100)
    df["gross_revenue"] = df["quantity"] * df["unit_price"]
    df["discount_amount"]= df["gross_revenue"] - df["revenue"]
    df["cost"]          = df["revenue"] * np.random.uniform(0.55, 0.75, len(df))
    df["profit"]        = df["revenue"] - df["cost"]
    df["profit_margin"] = (df["profit"] / df["revenue"].replace(0,np.nan) * 100).round(2).fillna(0)
    df["order_year"]    = df["order_date"].dt.year.fillna(2024).astype(int)
    df["order_month"]   = df["order_date"].dt.month.fillna(1).astype(int)
    df["order_quarter"] = df["order_date"].dt.quarter.fillna(1).astype(int)
    df["order_dow"]     = df["order_date"].dt.dayofweek.fillna(0).astype(int)
    df["is_weekend"]    = (df["order_dow"] >= 5).astype(int)
    df["is_high_value"] = (df["revenue"] > df["revenue"].quantile(0.75)).astype(int)
    df["has_discount"]  = (df["discount_pct"] > 0).astype(int)
    df["age_group"] = pd.cut(df["age"],bins=[0,25,35,45,60,100],
                              labels=["Gen Z","Millennial","Gen X","Boomer","Senior"]).astype(str)
    df["price_tier"] = pd.cut(df["unit_price"],bins=[0,500,2000,10000,1e9],
                               labels=["Budget","Mid-range","Premium","Luxury"]).astype(str)
    print(" Feature engineering done")

    # --- Encoding ---
    for col in ["category","payment_method","status","gender","age_group","price_tier","city"]:
        if col in df.columns:
            le = LabelEncoder()
            df[f"{col}_encoded"] = le.fit_transform(df[col].astype(str))
    print("  Encoding done")

    # --- Scaling ---
    for col in ["quantity","unit_price","discount_pct","revenue","profit","profit_margin","age"]:
        if col in df.columns:
            sc = StandardScaler()
            df[f"{col}_scaled"] = sc.fit_transform(df[[col]].fillna(0))
    print("   Scaling done")
    print(f"  Final shape: {df.shape}")
    return df


# ══════════════════════════════════════════════════════════════
# STAGE 4 — PySpark KPI Analytics
# ══════════════════════════════════════════════════════════════

def run_spark_pipeline(df_pandas):
    print(f"\n{'='*60}")
    print("  STAGE 4 — PySpark Distributed KPI Analytics...")
    print(f"{'='*60}")
    from pyspark.sql import SparkSession
    from pyspark.sql import functions as F

    spark = (SparkSession.builder
             .appName("Day10_Pipeline")
             .master("local[*]")
             .config("spark.driver.memory","2g")
             .config("spark.sql.shuffle.partitions","4")
             .config("spark.ui.showConsoleProgress","false")
             .getOrCreate())
    spark.sparkContext.setLogLevel("ERROR")

    core = ["order_id","customer_id","age","gender","city","category","product_name",
            "quantity","unit_price","discount_pct","order_date","payment_method",
            "status","rating","revenue","profit","profit_margin",
            "order_month","order_quarter","is_weekend","is_high_value","age_group","price_tier"]
    core = [c for c in core if c in df_pandas.columns]

    num_cols = ["quantity","unit_price","discount_pct","revenue","profit","profit_margin",
                "rating","order_month","order_quarter","is_weekend","is_high_value"]
    df_work = df_pandas[core].copy()
    for c in num_cols:
        if c in df_work: df_work[c] = pd.to_numeric(df_work[c], errors="coerce").fillna(0)
    for c in df_work.select_dtypes("object").columns:
        df_work[c] = df_work[c].astype(str).fillna("Unknown")

    df = spark.createDataFrame(df_work)
    print(f"  Loaded {df.count()} rows into Spark")

    kpis = {}

    kpis["overall"] = df.agg(
        F.count("order_id").alias("total_orders"),
        F.round(F.sum("revenue"),2).alias("total_revenue"),
        F.round(F.avg("revenue"),2).alias("avg_order_value"),
        F.round(F.sum("profit"),2).alias("total_profit"),
        F.round(F.avg("profit_margin"),2).alias("avg_profit_margin"),
        F.round(F.avg("rating"),2).alias("avg_rating"),
        F.countDistinct("customer_id").alias("unique_customers"),
        F.round(F.avg("discount_pct"),2).alias("avg_discount_pct")
    ).collect()[0].asDict()

    kpis["by_category"] = [r.asDict() for r in
        df.groupBy("category").agg(
            F.count("order_id").alias("orders"),
            F.round(F.sum("revenue"),2).alias("total_revenue"),
            F.round(F.avg("revenue"),2).alias("avg_order_value"),
            F.round(F.sum("profit"),2).alias("total_profit"),
            F.round(F.avg("profit_margin"),2).alias("avg_profit_margin"),
            F.round(F.avg("rating"),2).alias("avg_rating")
        ).orderBy(F.desc("total_revenue")).collect()]

    kpis["monthly_trend"] = [r.asDict() for r in
        df.groupBy("order_month").agg(
            F.count("order_id").alias("orders"),
            F.round(F.sum("revenue"),2).alias("revenue"),
            F.round(F.sum("profit"),2).alias("profit")
        ).orderBy("order_month").collect()]

    kpis["by_payment"] = [r.asDict() for r in
        df.groupBy("payment_method").agg(
            F.count("order_id").alias("orders"),
            F.round(F.sum("revenue"),2).alias("revenue")
        ).orderBy(F.desc("orders")).collect()]

    kpis["by_status"] = [r.asDict() for r in
        df.groupBy("status").agg(F.count("order_id").alias("count"))
        .orderBy(F.desc("count")).collect()]

    kpis["top_cities"] = [r.asDict() for r in
        df.groupBy("city").agg(
            F.round(F.sum("revenue"),2).alias("revenue"),
            F.count("order_id").alias("orders")
        ).orderBy(F.desc("revenue")).limit(10).collect()]

    kpis["quarterly"] = [r.asDict() for r in
        df.groupBy("order_quarter").agg(
            F.count("order_id").alias("orders"),
            F.round(F.sum("revenue"),2).alias("revenue"),
            F.round(F.avg("profit_margin"),2).alias("avg_margin")
        ).orderBy("order_quarter").collect()]

    spark.stop()
    print("  KPIs computed, Spark stopped")
    return kpis


# ══════════════════════════════════════════════════════════════
# STAGE 5 — Report Generation
# ══════════════════════════════════════════════════════════════

def _chart_style():
    plt.rcParams.update({
        "figure.facecolor":"#0F172A","axes.facecolor":"#1E293B",
        "axes.edgecolor":"#334155","axes.labelcolor":"#CBD5E1",
        "xtick.color":"#94A3B8","ytick.color":"#94A3B8",
        "text.color":"#E2E8F0","grid.color":"#334155",
        "grid.linestyle":"--","grid.alpha":0.5
    })

def make_charts(kpis):
    _chart_style()
    COLORS = ["#38BDF8","#818CF8","#34D399","#FB923C","#F472B6"]
    charts = {}

    # Category Revenue
    data = pd.DataFrame(kpis["by_category"])
    fig, ax = plt.subplots(figsize=(8,4))
    ax.barh(data["category"], data["total_revenue"], color=COLORS[:len(data)], edgecolor="none")
    ax.set_title("Revenue by Category", fontsize=13, fontweight="bold", color="#F8FAFC", pad=10)
    ax.set_xlabel("Revenue (Rs.)", fontsize=10)
    plt.tight_layout()
    p = "outputs/charts/category_revenue.png"
    plt.savefig(p, dpi=120, bbox_inches="tight", facecolor="#0F172A"); plt.close(); charts["category_revenue"] = p

    # Monthly Trend
    data = pd.DataFrame(kpis["monthly_trend"])
    MONTHS = ["Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep","Oct","Nov","Dec"]
    data["mn"] = data["order_month"].apply(lambda x: MONTHS[int(x)-1] if 1<=int(x)<=12 else str(x))
    fig, ax = plt.subplots(figsize=(9,4))
    ax.fill_between(data["mn"], data["revenue"], alpha=0.15, color="#38BDF8")
    ax.plot(data["mn"], data["revenue"], color="#38BDF8", lw=2.5, marker="o", ms=5, label="Revenue")
    ax.fill_between(data["mn"], data["profit"], alpha=0.15, color="#34D399")
    ax.plot(data["mn"], data["profit"], color="#34D399", lw=2, ls="--", marker="s", ms=4, label="Profit")
    ax.set_title("Monthly Revenue & Profit Trend", fontsize=13, fontweight="bold", color="#F8FAFC", pad=10)
    ax.legend(facecolor="#1E293B", edgecolor="#334155", labelcolor="#CBD5E1")
    plt.tight_layout()
    p = "outputs/charts/monthly_trend.png"
    plt.savefig(p, dpi=120, bbox_inches="tight", facecolor="#0F172A"); plt.close(); charts["monthly_trend"] = p

    # Payment Pie
    data = pd.DataFrame(kpis["by_payment"])
    fig, ax = plt.subplots(figsize=(5,5))
    wedges, texts, autotexts = ax.pie(data["orders"], labels=data["payment_method"],
        colors=COLORS[:len(data)], autopct="%1.1f%%", startangle=140, pctdistance=0.82,
        wedgeprops={"edgecolor":"#0F172A","linewidth":2})
    for t in texts: t.set_color("#CBD5E1")
    for a in autotexts: a.set_color("#0F172A"); a.set_fontsize(8)
    ax.set_title("Payment Methods", fontsize=12, fontweight="bold", color="#F8FAFC")
    plt.tight_layout()
    p = "outputs/charts/payment.png"
    plt.savefig(p, dpi=120, bbox_inches="tight", facecolor="#0F172A"); plt.close(); charts["payment"] = p

    # Top Cities
    data = pd.DataFrame(kpis["top_cities"]).head(8)
    fig, ax = plt.subplots(figsize=(8,4))
    ax.bar(data["city"], data["revenue"], color="#818CF8", edgecolor="none", width=0.6)
    ax.set_title("Top Cities by Revenue", fontsize=13, fontweight="bold", color="#F8FAFC", pad=10)
    ax.set_ylabel("Revenue (Rs.)", fontsize=10)
    plt.xticks(rotation=30, ha="right")
    plt.tight_layout()
    p = "outputs/charts/top_cities.png"
    plt.savefig(p, dpi=120, bbox_inches="tight", facecolor="#0F172A"); plt.close(); charts["top_cities"] = p

    # Order Status
    data = pd.DataFrame(kpis["by_status"])
    STATUS_COLORS = {"Delivered":"#34D399","Pending":"#FB923C","Cancelled":"#F87171","Returned":"#FBBF24"}
    fig, ax = plt.subplots(figsize=(6,3.5))
    ax.bar(data["status"], data["count"],
           color=[STATUS_COLORS.get(s,"#94A3B8") for s in data["status"]], edgecolor="none", width=0.5)
    ax.set_title("Order Status Distribution", fontsize=12, fontweight="bold", color="#F8FAFC")
    for i,(s,c) in enumerate(zip(data["status"],data["count"])):
        ax.text(i, c+0.5, str(c), ha="center", fontsize=9, color="#94A3B8")
    plt.tight_layout()
    p = "outputs/charts/status.png"
    plt.savefig(p, dpi=120, bbox_inches="tight", facecolor="#0F172A"); plt.close(); charts["status"] = p

    print(f" {len(charts)} charts saved")
    return charts

def generate_insights(kpis):
    print("  Generating LLM insights...")
    resp = client.messages.create(
        model="claude-sonnet-4-20250514", max_tokens=700,
        messages=[{"role":"user","content":
            f"""You are a senior data analyst. Write a concise executive summary (200-250 words) based on these KPIs.

KPIs: {json.dumps(kpis, default=str)}

Cover: overall performance, top category, revenue trends, 2 recommendations. Use flowing paragraphs, no bullets."""}]
    )
    return resp.content[0].text

class ReportPDF(FPDF):
    def header(self):
        self.set_fill_color(15,23,42); self.rect(0,0,210,297,"F")
        self.set_fill_color(30,41,59); self.rect(0,0,210,20,"F")
        self.set_font("Helvetica","B",12); self.set_text_color(248,250,252)
        self.set_xy(0,5); self.cell(210,10,"DAY 10 - AI Agent for Automated Data Engineering",align="C")
    def footer(self):
        self.set_y(-13); self.set_font("Helvetica","",8); self.set_text_color(100,116,139)
        self.cell(0,10,f"End-to-End Agentic Data Pipeline  |  Page {self.page_no()}",align="C")
    def section(self, title):
        self.ln(5)
        self.set_fill_color(30,41,59); self.set_font("Helvetica","B",11); self.set_text_color(56,189,248)
        self.cell(0,8,f"  {title}",fill=True,ln=True,border="L")
        self.ln(2)

def build_pdf(kpis, insights, charts, path="outputs/kpi_report.pdf"):
    overall = kpis["overall"]
    pdf = ReportPDF(); pdf.set_auto_page_break(True,15); pdf.add_page()

    pdf.ln(25)
    pdf.set_font("Helvetica","B",20); pdf.set_text_color(248,250,252)
    pdf.cell(0,10,"Retail Analytics KPI Report",align="C",ln=True)
    pdf.set_font("Helvetica","",10); pdf.set_text_color(148,163,184)
    pdf.cell(0,7,"Generated by End-to-End Agentic Data Engineering Pipeline",align="C",ln=True)
    pdf.ln(5)

    # KPI Cards
    cards = [
        ("Total Orders",     f"{int(overall.get('total_orders',0)):,}"),
        ("Total Revenue",    f"Rs.{float(overall.get('total_revenue',0)):,.0f}"),
        ("Total Profit",     f"Rs.{float(overall.get('total_profit',0)):,.0f}"),
        ("Avg Order Value",  f"Rs.{float(overall.get('avg_order_value',0)):,.0f}"),
        ("Avg Margin",       f"{float(overall.get('avg_profit_margin',0)):.1f}%"),
    ]
    y0 = pdf.get_y(); cw = 38
    for i,(lbl,val) in enumerate(cards):
        x = 10 + i*(cw+2)
        pdf.set_xy(x,y0); pdf.set_fill_color(30,41,59); pdf.set_draw_color(51,65,85)
        pdf.rect(x,y0,cw,18,"DF")
        pdf.set_xy(x,y0+2); pdf.set_font("Helvetica","",7); pdf.set_text_color(148,163,184)
        pdf.cell(cw,5,lbl,align="C")
        pdf.set_xy(x,y0+8); pdf.set_font("Helvetica","B",10); pdf.set_text_color(56,189,248)
        pdf.cell(cw,7,val,align="C")
    pdf.ln(24)

    pdf.section("Executive Insights")
    pdf.set_font("Helvetica","",10); pdf.set_text_color(203,213,225)
    pdf.multi_cell(0,6,insights)

    pdf.section("Revenue by Category")
    if os.path.exists(charts["category_revenue"]): pdf.image(charts["category_revenue"],x=10,w=185)

    pdf.section("Monthly Revenue & Profit Trend")
    if os.path.exists(charts["monthly_trend"]): pdf.image(charts["monthly_trend"],x=10,w=185)

    pdf.add_page(); pdf.ln(25)
    pdf.section("Payment Methods & Order Status")
    y_now = pdf.get_y()
    if os.path.exists(charts["payment"]): pdf.image(charts["payment"],x=10,y=y_now,w=90)
    if os.path.exists(charts["status"]): pdf.image(charts["status"],x=108,y=y_now,w=92)
    pdf.ln(85)

    pdf.section("Top Cities by Revenue")
    if os.path.exists(charts["top_cities"]): pdf.image(charts["top_cities"],x=10,w=185)

    pdf.section("Category Performance Table")
    hdrs = ["Category","Orders","Revenue","Profit","Margin%","Rating"]
    ws   = [40,22,36,36,24,28]
    pdf.set_fill_color(30,41,59); pdf.set_font("Helvetica","B",8); pdf.set_text_color(56,189,248)
    for h,w in zip(hdrs,ws): pdf.cell(w,7,h,border=1,align="C",fill=True)
    pdf.ln()
    pdf.set_font("Helvetica","",8)
    for i,row in enumerate(kpis["by_category"]):
        pdf.set_fill_color(15,23,42) if i%2==0 else pdf.set_fill_color(24,33,52)
        pdf.set_text_color(203,213,225)
        for val,w in zip([
            str(row.get("category","")), str(row.get("orders","")),
            f"Rs.{float(row.get('total_revenue',0)):,.0f}",
            f"Rs.{float(row.get('total_profit',0)):,.0f}",
            f"{float(row.get('avg_profit_margin',0)):.1f}%",
            f"{float(row.get('avg_rating',0)):.2f}"
        ], ws):
            pdf.cell(w,6,val,border=1,align="C",fill=True)
        pdf.ln()

    pdf.output(path)
    print(f" PDF saved: {path}")
    return path

def build_markdown(kpis, insights, path="outputs/kpi_report.md"):
    o = kpis["overall"]
    MONTHS = ["Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep","Oct","Nov","Dec"]
    lines = [
        "# Day 10 — AI Agent Data Engineering Report\n",
        "## Overall KPIs\n",
        "| Metric | Value |","|----|----|",
        f"| Total Orders | {int(o.get('total_orders',0)):,} |",
        f"| Total Revenue | Rs.{float(o.get('total_revenue',0)):,.2f} |",
        f"| Total Profit | Rs.{float(o.get('total_profit',0)):,.2f} |",
        f"| Avg Order Value | Rs.{float(o.get('avg_order_value',0)):,.2f} |",
        f"| Avg Profit Margin | {float(o.get('avg_profit_margin',0)):.1f}% |",
        f"| Avg Rating | {float(o.get('avg_rating',0)):.2f} |",
        f"| Unique Customers | {int(o.get('unique_customers',0)):,} |",
        "","## Executive Insights\n", insights,
        "","## Revenue by Category\n",
        "| Category | Orders | Revenue | Profit | Margin% |",
        "|----------|--------|---------|--------|---------|",
    ]
    for r in kpis["by_category"]:
        lines.append(f"| {r['category']} | {r['orders']} | Rs.{float(r['total_revenue']):,.0f} | Rs.{float(r['total_profit']):,.0f} | {float(r['avg_profit_margin']):.1f}% |")
    lines += ["","## Monthly Trend\n","| Month | Orders | Revenue | Profit |","|-------|--------|---------|--------|"]
    for r in kpis["monthly_trend"]:
        m = MONTHS[int(r["order_month"])-1] if 1<=int(r["order_month"])<=12 else str(r["order_month"])
        lines.append(f"| {m} | {r['orders']} | Rs.{float(r['revenue']):,.0f} | Rs.{float(r['profit']):,.0f} |")
    lines += ["","---","_Generated by Day 10 Agentic Data Engineering Pipeline_"]
    with open(path,"w") as f: f.write("\n".join(lines))
    print(f"  Markdown saved: {path}")
    return path

def generate_report(kpis):
    print(f"\n{'='*60}")
    print("  STAGE 5 — Report Generation (PDF + Markdown)...")
    print(f"{'='*60}")
    charts   = make_charts(kpis)
    insights = generate_insights(kpis)
    pdf_path = build_pdf(kpis, insights, charts)
    md_path  = build_markdown(kpis, insights)
    return pdf_path, md_path


print("All pipeline functions defined — ready to run!")

In [ ]:
# ============================================================
# CELL 4 — Run the Full Pipeline
# ============================================================
import time

PROMPT      = "Generate a realistic Indian e-commerce retail sales dataset with diverse customers, products, and orders."
NUM_RECORDS = 200   # must be a multiple of 50

t0 = time.time()

# Stage 1
df_raw = generate_dataset(PROMPT, num_records=NUM_RECORDS)
df_raw.to_csv("outputs/raw_dataset.csv", index=False)

# Stage 2
df_clean = clean_dataset(df_raw)
df_clean.to_csv("outputs/cleaned_dataset.csv", index=False)

# Stage 3
df_transformed = transform_dataset(df_clean)
df_transformed.to_csv("outputs/transformed_dataset.csv", index=False)

# Stage 4
kpis = run_spark_pipeline(df_transformed)
with open("outputs/kpis.json", "w") as f:
    json.dump(kpis, f, indent=2, default=str)

# Stage 5
pdf_path, md_path = generate_report(kpis)

print(f"\n{'='*60}")
print(f"  PIPELINE COMPLETE in {time.time()-t0:.1f}s")
print(f"{'='*60}")
o = kpis['overall']
print(f"  Total Orders  : {int(o.get('total_orders',0)):,}")
print(f"  Total Revenue : Rs.{float(o.get('total_revenue',0)):,.2f}")
print(f"  Total Profit  : Rs.{float(o.get('total_profit',0)):,.2f}")
print(f"  Avg Rating    : {float(o.get('avg_rating',0)):.2f} / 5.0")
print(f"{'='*60}")

In [ ]:
# ============================================================
# CELL 5 — Display Charts Inline
# ============================================================
from IPython.display import Image, display
import glob

for chart_path in sorted(glob.glob("outputs/charts/*.png")):
    print(f"\n {chart_path}")
    display(Image(filename=chart_path, width=700))

In [ ]:
# ============================================================
# CELL 6 — Preview KPI Tables
# ============================================================
print("\n OVERALL KPIs")
display(pd.DataFrame([kpis['overall']]).T.rename(columns={0:'Value'}))

print("\n CATEGORY PERFORMANCE")
display(pd.DataFrame(kpis['by_category']))

print("\n TOP 10 CITIES")
display(pd.DataFrame(kpis['top_cities']))

print("\n PAYMENT METHODS")
display(pd.DataFrame(kpis['by_payment']))

In [ ]:
# ============================================================
# CELL 7 — Download Output Files
# ============================================================
from google.colab import files

print("Downloading generated files...")
files.download("outputs/kpi_report.pdf")
files.download("outputs/kpi_report.md")
files.download("outputs/raw_dataset.csv")
files.download("outputs/cleaned_dataset.csv")
files.download("outputs/transformed_dataset.csv")
files.download("outputs/kpis.json")
print(" All files downloaded!")